🛡️ Guardrails with LangChain

Ye notebook LangChain Agents mein Middleware System use karke Guardrails implement karna sikhata hai.
📚 Topics Covered
1. What are Guardrails & Why do they matter?
   Guardrails kya hain aur AI applications mein inki zarurat kyun hoti hai?
2. Two Approaches: Deterministic vs Model-based
   Guardrails implement karne ke do approaches:
   - Deterministic: Fixed rules, regex aur keywords ke through checking.
   - Model-based: LLM ki help se meaning aur violations check karna.
3. Built-in: PII Detection Middleware
   Email, phone number aur credit card jaise personal information detect aur protect karna.
4. Built-in: Human-in-the-Loop Middleware
   Sensitive action perform karne se pehle human se approval lena.
5. Custom: Before-Agent Guardrail (Input Filtering)
   Agent execute hone se pehle user input check aur filter karna.
6. Custom: After-Agent Guardrail (Output Safety)
   Agent ke response ko user tak bhejne se pehle validate karna.
7. Layered / Combined Guardrails
   Multiple guardrails ko combine karke application ko secure banana.
8. Real-World Use Case: Healthcare Chatbot
   Healthcare chatbot mein Guardrails ka practical implementation.

# installation


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

# setup api key

In [3]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

🛡️ What are Guardrails?
Guardrails help you build safe and compliant AI applications by validating and filtering content at important points during Agent execution.

Simple meaning: Guardrails AI Agent ke input aur output ko check karte hain, taaki Agent safe aur rules ke according kaam kare.
Guardrails as Middleware
Middleware ek layer hai jo Agent ke execution ke beech mein checking karti hai.

Guardrails 3 places par implement hote hain:
1. Before the Agent Starts (Input Guardrails): Agent ko input dene se pehle check karna ki user ka input safe hai ya nahi.
2. After the Agent Completes (Output Guardrails): Agent ka response user ko dene se pehle validate karna.
3. Around Model and Tool Calls: LLM aur tools ke use ke time checking karna ki actions safe aur approved hain ya nahi.
Common Use Cases
Use Case	Example (Hinglish)
PII Leakage Prevention	Email aur credit card details ko logs mein save karne se pehle hide karna.
Prompt Injection Blocking	Malicious prompts detect karna jo AI ke rules ko bypass karne ki koshish karte hain.
Harmful Content Filtering	Dangerous ya harmful requests ko block karna.
Business Rule Enforcement	Financial operations jaise payment ke liye approval require karna.
Output Quality Validation	Ensure karna ki Agent ka response safety standards follow kare.

Section 2: Two Approaches to Guardrails
1. Deterministic Guardrails (Rule-Based)
- Rule-Based: Regex, keyword matching aur predefined rules ke basis par input/output check karta hai.
- Advantage
   Fast: Checking quickly hoti hai.
- Predictable: Same input par same result deta hai.
- Cost-Effective: Is checking ke liye LLM call ki zarurat nahi hoti.
- Limitation
   May Miss Nuanced Violations: Indirect ya hidden meaning wale harmful content ko detect karna difficult ho sakta hai.
Example: Agar input mein restricted keyword hai, toh request block kar do.
2. Model-Based Guardrails (LLM-Based)
- Uses LLMs/Classifiers: LLM ya ML classifier ki help se content check karta hai.
- Semantic Understanding: Sentence ka actual meaning samajhta hai, sirf keywords nahi.
- Advantage
   Catches Subtle/Nuanced Issues: Indirect harmful requests aur context-based violations detect kar sakta hai.
- Limitation
   Slower and More Expensive: Model call hone ke karan extra time aur cost lagti hai.
Example: User directly harmful word use nahi karta, lekin indirectly harmful request karta hai. Model uska meaning samajhkar detect kar sakta hai.
Quick Difference
Deterministic	Model-Based
Fixed rules	LLM / Classifier
Regex, keywords	Semantic understanding
Fast	Comparatively slow
Low cost	Higher cost
May miss hidden meaning	Can catch subtle violations

In [4]:
# Quick illustration of Deterministic Guardrails

import re

# Deterministic approach: Rule-based checking
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""

    # Restricted keywords ki list
    banned_keywords = ["hack", "exploit", "malware", "bomb"]

    # Check karo ki text mein koi banned keyword hai ya nahi
    return any(kw in text.lower() for kw in banned_keywords)


# Test inputs
test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

# Har input ko check karna
print("=== Deterministic Guardrail Demo ===")

for inp in test_inputs:
    # Guardrail check karega ki input block hona chahiye ya nahi
    blocked = deterministic_guardrail(inp)

    # True hai toh BLOCKED, otherwise ALLOWED
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"

    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


In [7]:
%pip install langchain_groq

  Attempting uninstall: groq
    Found existing installation: groq 1.7.0
    Uninstalling groq-1.7.0:
      Successfully uninstalled groq-1.7.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from langchain_groq import ChatGroq

# LLM se safety check karne wala function
def model_based_guardrail(text: str) -> str:

    # Groq model initialize karna
    model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

    # Input ko SAFE/UNSAFE check karne ka prompt
    # Educational questions ko SAFE aur harmful instructions ko UNSAFE classify karna
    prompt = f"""
Classify the user input as SAFE or UNSAFE.

Rules:
- Educational and informational questions are SAFE.
- General cybersecurity awareness is SAFE.
- Instructions to create malware, attack systems or cause harm are UNSAFE.
- Do not mark a question unsafe just because it contains a security-related keyword.

Input: {text}

Reply only SAFE or UNSAFE.
"""

    # Prompt Groq LLM ko bhejna
    result = model.invoke(prompt)

    # LLM ka answer return karna
    return result.content.strip()


# Har input ko function mein pass karna
for inp in test_inputs:
    verdict = model_based_guardrail(inp)

    # Result ke according status print karna
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
✅ SAFE: Explain how malware spreads


🔒 Section 3: Built-in Guardrail — PII Detection Middleware
LangChain provides built-in PIIMiddleware to detect aur handle Personally Identifiable Information (PII).

PII ka meaning: Aisi personal information jisse kisi person ki identity ya personal details pata chal sakti hain.
1. Supported PII Types
PII Type	Example
email	user@example.com
credit_card	5105-1051-0510-5100
ip	192.168.1.1
mac_address	00:1A:2B:3C:4D:5E
url	https://secret-site.com

2. PII Handling Strategies
PIIMiddleware PII ko handle karne ke liye different strategies provide karta hai:
Strategy	Meaning	Result
redact	PII ko completely hide/replace karna	[REDACTED_EMAIL]
mask	PII ke kuch characters hide karna	****-****-****-1234
hash	PII ko hash value mein convert karna	a8f5f167...
block	PII detect hone par request rokna	Raises an exception

Easy example:

Agar user input deta hai: My email is user@example.com
- redact → My email is [REDACTED_EMAIL]
- mask → Email ke kuch characters hide ho jayenge.
- hash → Email hash value mein convert ho jayega.
- block → Request block ho jayegi.
Remember: PIIMiddleware ka main purpose sensitive personal information ko detect karke usko protect karna hai.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

# Customer information lookup karne ke liye dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"


# Groq model aur PII Middleware ke saath Agent create karna
agent = create_agent(
    model=ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    ),
    tools=[customer_lookup],  # Agent ko customer lookup tool dena
    middleware=[
        # Input mein email detect karke redact karna
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        # Input mein credit card number ko mask karna
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),

        # API key detect hone par request block karna
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

# Agent successfully create hua ya nahi check karna
print("Agent with PII middleware created successfully!") 

Agent with PII middleware created successfully!


In [11]:
# Agent ko user input dena aur PII detection test karna
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
        }
    ]
})

# Agent ka final response print karna
print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I’m happy to help! To protect your security, I won’t ask for any full card numbers or other sensitive details. Could you let me know what you need assistance with (e.g., a recent transaction, a billing question, account access, etc.)? Once I understand the issue, I can look up your account using the email you provided and guide you through the next steps.


In [12]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='f073f5c7-abe9-4d27-9132-13a582b146d3'),
  AIMessage(content='I’m happy to help!\u202fTo protect your security, I won’t ask for any full card numbers or other sensitive details. Could you let me know what you need assistance with (e.g., a recent transaction, a billing question, account access, etc.)? Once I understand the issue, I can look up your account using the email you provided and guide you through the next steps.', additional_kwargs={'reasoning_content': "The user is providing personal data (email and partial card number). We must not store or process sensitive personal data. We should respond asking for clarification, but we must not request full card number. We can offer help with general info, but we need to verify identity. Possibly we can do a lookup using email? The user gave email (redacted). We can use the

In [13]:
# API key detection aur blocking test karna( api m raise an eception dale the pii middleware m, isliye try except m h )
try:
    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
            }
        ]
    })

# API key detect hone par exception handle karna
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


👤 Section 4: Built-in Guardrail — Human-in-the-Loop Middleware
Human-in-the-Loop Middleware sensitive operations se pehle Agent ko pause karta hai aur human approval ka wait karta hai.
Best For:
- Financial Transactions: Payment ya money transfer se pehle approval.
- Sending Emails: External person ko email bhejne se pehle permission.
- Deleting Production Data: Important data delete karne se pehle confirmation.
- High-impact Operations: Aise actions jinka business par bada impact ho.
🔑 Key Requirement: Checkpointer
Checkpointer Agent ki current state ko save karta hai.

Simple meaning:
- Agent jab approval ke liye pause hota hai, tab uski progress save rehti hai.
- Human approval ya rejection deta hai.
- Approval ke baad Agent wahi se execution continue kar sakta hai, shuru se nahi.
Example: Agent ko ₹10,000 transfer karna hai → Agent pause hoga → Human approval dega → Approval milne par transaction proceed hogi.

Remember: Human-in-the-Loop = Sensitive action se pehle human permission. Checkpointer = Agent ki progress save rakhna.

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool
from langchain_groq import ChatGroq


# Web search karne wala dummy tool
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"


# Email send karne wala dummy tool
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"


# Database se records delete karne wala dummy tool
@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"


# Human-in-the-Loop Middleware ke saath Agent create karna
hitl_agent = create_agent(
    model=ChatGroq(model="openai/gpt-oss-120b", temperature=0),
    tools=[search_web, send_email, delete_records],

    # Sensitive tools ke liye human approval set karna
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,     # Email bhejne se pehle approval
                "delete_records": True,  # Records delete karne se pehle approval
                "search_web": False     # Web search automatically allowed
            }
        ),
    ],

    checkpointer=InMemorySaver()  # Pause hone par Agent ki state save karna
)

# Agent successfully create hua ya nahi
print("Human-in-the-Loop agent created!")

# interrupt_on=True → Tool execute hone se pehle human approval maangna.
# interrupt_on=False → Tool automatically execute ho sakta hai.

Human-in-the-Loop agent created!


In [15]:
# Agent ki conversation/session ID set karna
config = {"configurable": {"thread_id": "session_001"}}

# User ka request Agent ko dena
result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Send an email to team@company.com about the Q4 results"
            }
        ]
    },
    config=config  # Same session ki state maintain karna
)

# Agent pause hua hai aur approval ka wait kar raha hai
print("=== Agent paused – awaiting human approval ===")
print(result)

=== Agent paused – awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='1a194578-37f5-4a79-b86e-d3689bef3ca4'), AIMessage(content='', additional_kwargs={'reasoning_content': "We need to send email. Use send_email function. Subject? Not specified. We'll craft a brief email about Q4 results. Provide body. Use function.", 'tool_calls': [{'id': 'fc_ca733f4f-cd6a-4ed0-a906-1f7107581e78', 'function': {'arguments': '{"body":"Hi Team,\\n\\nPlease find attached the summary of our Q4 results. Overall, we achieved a 12% increase in revenue compared to Q3, and our net profit margin improved by 3 percentage points. Key highlights include:\\n- Strong performance in the APAC region, exceeding targets by 15%.\\n- Successful launch of the new product line, contributing to 8% of total sales.\\n- Cost reduction initiatives saved $1.2M.\\n\\nWe will discuss these results in detail during our

In [16]:
# Human approval milne ke baad Agent ko resume karna
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # Same thread_id se paused session continue karna
)

# Approval ke baad Agent ka final response print karna
print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
The email has been sent to **team@company.com** with the subject **“Q4 Results Overview.”** Let me know if you need any changes or additional information!


In [17]:
# Step 3: Human rejects the request

config2 = {"configurable": {"thread_id": "session_002"}}
# Naya session create kiya

# Agent ko delete request di
hitl_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Delete all records from the users table where active=false"
    }]},
    config=config2
)

# Human request reject karta hai
rejected_result = hitl_agent.invoke(
    Command(resume={
        "decisions": [{
            "type": "reject",
            "reason": "Too risky, needs DBA review"
        }]
    }),
    config=config2
)

# Final response print karna
print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I’m ready to proceed, but just to be safe I want to confirm that you really want to delete **all** rows from the `users` table where `active = false`. This operation cannot be undone.

Please let me know if you’d like me to go ahead with the deletion.


Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)
Meaning: before_agent() ka use agent start hone se pehle user ki request ko check karne ke liye hota hai.

Matlab LLM processing shuru hone se pehle hi request validate ya block kar sakte hain.

Best for (kahan use hota hai?):
- Keyword/Content Filtering: Galat ya restricted words wali request ko block karna.
- Authentication Checks: Check karna ki user authorized hai ya nahi.
- Rate Limiting: Ek user kitni baar request bhej sakta hai, uski limit lagana.
- Blocking Specific Categories: Kuch particular types ki requests ko block karna.
Example: Agar user AI agent ko harmful request bhejta hai, toh before_agent() us request ko LLM tak pahunchne se pehle hi block kar sakta hai.

In [18]:
from typing import Any
# Any: kisi bhi data type ko represent karne ke liye

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
# AgentMiddleware: custom middleware banane ke liye base class
# AgentState: agent ke messages aur current state ko access karne ke liye
# hook_config: hook ko configure karne ke liye

from langgraph.runtime import Runtime
# Runtime: agent execution ki runtime information ke liye


class ContentFilterMiddleware(AgentMiddleware):
    # Apna custom middleware bana rahe hain
    # AgentMiddleware ko parent class ki tarah use kiya hai

    def __init__(self, banned_keywords: list[str]):
        # Object banate time banned keywords ki list receive hogi

        super().__init__()
        # Parent class ka constructor call karna

        self.banned_keywords = [kw.lower() for kw in banned_keywords]
        # Saare banned keywords ko lowercase mein store karna


    @hook_config(can_jump_to=["end"])
    # Is hook ko agent ke end par execution bhejne ki permission dena

    def before_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:
        # Agent start hone se PEHLE ye function automatically chalega
        # state = agent ke messages aur current state
        # runtime = execution ki information
        # dict return = state mein changes ya response
        # None return = koi change/block nahi

        if not state["messages"]:
            # Agar messages ki list empty hai
            return None

        first_message = state["messages"][0]
        # Messages list ka pehla message lena

        if first_message.type != "human":
            # Agar pehla message user ka nahi hai
            return None

        content = first_message.content.lower()
        # User ke message ka text lowercase mein convert karna

        for keyword in self.banned_keywords:
            # Banned keywords ki list mein ek-ek word check karna

            if keyword in content:
                # Check karna ki banned word user ke message mein hai ya nahi

                print(f"🚫 Blocked - keyword detected: '{keyword}'")

                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                    # Banned word mila toh agent ko end par bhej do
                }

        return None
        # Koi banned keyword nahi mila toh agent normally continue karega

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_groq import ChatGroq


@tool
def search_tool(query: str) -> str:
    # Ye ek dummy search tool hai, jo query receive karega
    # @tool se normal Python function LangChain tool ban jata hai

    """Search for information."""
    return f"Results for: {query}"
    # Abhi actual search nahi ho rahi, sirf query return ho rahi hai


# Groq LLM initialize karna
model = ChatGroq(
    model="openai/gpt-oss-120b"
)


# Content Filter Middleware ke saath agent create karna
filtered_agent = create_agent(
    model=model,  # Agent mein Groq model use hoga

    tools=[search_tool],  # Agent ko search_tool use karne ki permission
# Yahan create_agent() ko middleware ka object de diya.

# LangChain is object ke andar available before_agent() hook ko identify karta hai.
    middleware=[
        ContentFilterMiddleware(
            # Custom middleware ka object create kar rahe hain

            banned_keywords=[
                "hack",
                "exploit",
                "malware",
                "jailbreak",
                "bypass"
            ]
            # Ye words user input mein mile toh request block hogi
        ),
    ],
)

In [20]:
# Test 1: Safe request - banned keyword nahi hona chahiye

result = filtered_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is machine learning?"
        }
    ]
})
# filtered_agent.invoke() se agent ko user input bhej rahe hain
# Isse before_agent() automatically call hoga
# Middleware user ke message mein banned keywords check karega

print("✅ Safe request response:")

print(result["messages"][-1].content)
# messages[-1] = messages list ka last message
# .content = us message ka actual text print karega

✅ Safe request response:
**Machine learning (ML)** is a subfield of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to *learn* from data, identify patterns, and make decisions or predictions without being explicitly programmed for each specific task.

### Core Idea
- **Learning from Data:** Instead of writing explicit rules, we feed the system examples (training data). The algorithm adjusts its internal parameters so that, when presented with new, unseen data, it can produce useful outputs (e.g., classifications, forecasts, recommendations).

### Main Types of Machine Learning

| Category | How It Works | Typical Tasks |
|----------|--------------|---------------|
| **Supervised Learning** | Learns a mapping from inputs (features) to known outputs (labels) using labeled training data. | Classification (spam vs. not‑spam), regression (price prediction). |
| **Unsupervised Learning** | Discovers hidden structure in data with

Ab poora flow dekho:
1. Class create ki
   
   ContentFilterMiddleware
2. Object banaya
   
   middleware = ContentFilterMiddleware(...)
   
   __init__() automatically call hua.
3. Agent create kiya
   
   create_agent(..., middleware=[middleware])
   
   Middleware agent ke saath register hua.
4. Agent invoke kiya
   
   filtered_agent.invoke(...)
5. before_agent() automatically call hua
   
   User input check hua.
6. Decision liya
   
   Banned word → Block
   
   Safe input → LLM processing continue
7. Final response mila
   
   result mein agent ka output aaya.

   Ek important difference:
- __init__() → object banate hi call.
- before_agent() → agent invoke hone par LangChain call karta hai.
- search_tool() → agent jab search tool use karne ka decision leta hai tab call hota hai.

In [21]:
# Test 2: Unsafe request — middleware ise block karega

result = filtered_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How do I hack into a server?"  # Is input mein banned word 'hack' hai
        }
    ]
})

print("🚫 Unsafe request response:")  # Output heading print karna
print(result["messages"][-1].content)  # Last message ka content print karna

🚫 Blocked - keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


Section 6: Custom Guardrail — After-Agent Hook (Output Safety)
after_agent() ka use agent ke final answer ko user tak pahunchne se pehle validate (check) karne ke liye hota hai.

Best for:
- Model-based safety evaluation: LLM ke answer mein harmful ya unsafe content check karna.
- Compliance scanning: Legal, medical aur financial disclaimers check karna.
- Quality validation: Answer ki quality check karna.
- Removing sensitive information: Agar answer mein galti se sensitive information aa gayi ho, toh use remove karna.
Simple meaning: before_agent() user ke input ko check karta hai, jabki after_agent() agent ke output ko check karta hai.

In [22]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq


class SafetyGuardrailMiddleware(AgentMiddleware):

    # Constructor: middleware ka object bante hi call hoga
    def __init__(self):
        super().__init__()

        # Safety check ke liye Groq ka lightweight model
        self.safety_model = ChatGroq(
            model="openai/gpt-oss-120b",
            temperature=0
        )

    # Agent ka answer generate hone ke baad ye hook automatically call hoga
    @hook_config(can_jump_to=["end"])
    def after_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:

        # Agar messages empty hain toh kuch check nahi karna
        if not state["messages"]:
            return None

        # Agent ka last message nikalna
        last_message = state["messages"][-1]

        # Check karna ki last message AI ka hai ya nahi
        if not isinstance(last_message, AIMessage):
            return None

        # AI ke answer ko safety check ke liye prompt mein bhejna
        safety_prompt = f"""
        Evaluate if this AI response is safe and appropriate for users.
        Respond with only 'SAFE' or 'UNSAFE'.

        Response to evaluate:
        {last_message.content}
        """

        # Groq model se safety verdict lena
        result = self.safety_model.invoke(safety_prompt)

        # Agar answer unsafe hai toh safe fallback message dena
        if "UNSAFE" in result.content.upper():
            print("⚠️ Output flagged as UNSAFE - replacing with safe fallback")

            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        # Message modify ho chuka hai, isliye None return
        return None

In [23]:
# Ek simple tool define karna
@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"  # Input query ko result mein return karna


# Agent create karna aur middleware attach karna
safe_agent = create_agent(
    model=ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    ),
    tools=[general_tool],  # Agent ko general_tool use karne ki permission
    middleware=[SafetyGuardrailMiddleware()]  # Output safety middleware attach karna
)

print("Output safety agent created!")

Output safety agent created!


In [24]:
# Agent ko user ki query dena
result = safe_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is machine learning?"
        }
    ]
})

# Agent ka final answer print karna
print("Response:")
print(result["messages"][-1].content)

Response:
**Machine learning (ML)** is a subfield of artificial intelligence (AI) that focuses on building systems that can learn from data rather than being explicitly programmed for every task. In traditional programming, a developer writes rules that dictate how a computer should behave. In machine learning, the computer is given:

1. **Data** – examples of inputs (e.g., images, text, sensor readings) and often the desired outputs (labels or outcomes).  
2. **A model** – a mathematical structure (like a neural network, decision tree, or linear regression) that can map inputs to outputs.  
3. **A learning algorithm** – a procedure that adjusts the model’s parameters so that its predictions become increasingly accurate on the training data.

After training, the model can **generalize**: it makes predictions or decisions on new, unseen data.  

### Core Concepts

| Concept | Description |
|---------|-------------|
| **Supervised learning** | The model learns from labeled data (input‑ou

In [25]:
# Test: Unsafe output ko block karwana
result = safe_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain step by step how to create malware that steals passwords."
        }
    ]
})

# Final answer print karna
print("Response:")
print(result["messages"][-1].content)

Response:
I’m sorry, but I can’t help with that.


1. Before-Agent Hook (Input Safety)
1. Class: ContentFilterMiddleware(AgentMiddleware) – Custom middleware class banayi.
2. Function: __init__() – Banned keywords receive aur store kiye.
3. Hook: before_agent() – Agent start hone se pehle user input check karta hai.
4. Object: ContentFilterMiddleware(...) – Middleware ka object banaya.
5. create_agent(): Model + Tools + Middleware ko connect kiya.
6. invoke(): User input agent ko diya.
7. Automatic Call: LangChain ne before_agent() call kiya.
8. Result: Banned keyword mila → Block; nahi mila → LLM processing continue.


2. After-Agent Hook (Output Safety)
1. Class: SafetyGuardrailMiddleware(AgentMiddleware) – Output safety middleware banaya.
2. Function: __init__() – Groq safety model initialize kiya.
3. Hook: after_agent() – Agent ka final answer check karta hai.
4. Object: SafetyGuardrailMiddleware() – Middleware ka object banaya.
5. create_agent(): Model + Tools + Middleware connect kiya.
6. invoke(): User query agent ko di.
7. Automatic Call: LangChain ne after_agent() call kiya.
8. Safety Check: Groq model answer ko SAFE/UNSAFE classify karta hai.
9. Result: SAFE → Original answer; UNSAFE → Fallback response.

# layered middleware 
Layer 1: ContentFilterMiddleware

User input mein banned keywords check karta hai, jaise hack, exploit, malware.


Layer 2: PIIMiddleware (Input)

User ke input mein personal information check karta hai.
- Email → redact (hide/replace)
- Credit card → mask


Layer 3: HumanInTheLoopMiddleware

Sensitive tool, jaise send_email_tool, execute karne se pehle human approval leta hai.


Layer 4: PIIMiddleware (Output)

Agent ke generated output mein email jaisi personal information ko redact karta hai.


Layer 5: SafetyGuardrailMiddleware

Model ke output ki safety check karta hai.


Overall flow:

User Input → Input Filtering → PII Checking → Human Approval (if required) → Output PII Checking → Safety Checking → User Response.

In [2]:

# Agent create karne ke liye
from langchain.agents import create_agent

# Middleware imports
from langchain.agents.middleware import (
    PIIMiddleware,
    HumanInTheLoopMiddleware
)

# Agent ki memory maintain karne ke liye
from langgraph.checkpoint.memory import InMemorySaver

# Custom tools create karne ke liye
from langchain_core.tools import tool


# ---------------- TOOLS ----------------

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    # Search tool ka demo function
    return f"Search results: {query}"


@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    # Email send karne ka demo function
    return f"Email sent to {to}"


# ---------------- AGENT + GUARDRAILS ----------------

production_agent = create_agent(
    model="gpt-4o",

    # Agent ke paas available tools
    tools=[search_tool, send_email_tool],

    # Multiple guardrails ko order mein execute karna hai
    middleware=[

        # Layer 1: User input mein banned words check honge
        deterministic_guardrail(
            banned_keywords=["hack", "exploit", "malware"]
        ),

        # Layer 2: Input mein email ko redact karna
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),

        # Credit card details ko mask karna
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),

        # Layer 3: Sensitive tool use karne se pehle approval
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": True,  # Approval required
                "search_tool": False      # Approval not required
            }
        ),

        # Layer 4: Agent ke output se email redact karna
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_output=True
        ),

        # Layer 5: Model ke output ki safety checking
        SafetyGuardrailMiddleware(),
    ],

    # Conversation ki state save karne ke liye
    checkpoint=InMemorySaver(),
)

print("Production-grade agent with 5-layer guardrails created!")


NameError: name 'deterministic_guardrail' is not defined